In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import pickle
from src.data_loader import load_data_cleaned

In [3]:
df = load_data_cleaned("mattresses_clean.csv")
display(df)

,product_name,brand,material_type,category,description,rating,reviews,product_sold_number,image_url,link,origin,warranty,firmness,thickness,price,width,length
0,Nệm cao su thiên nhiên Gummi Classic thế hệ mớ...,Gummi,Cao su thiên nhiên,Cao su,"Thương hiệu Gummi - Tinh khiết từ cao su, đúng...",4.9,359.0,44410.0,"https://cdn-cf-ops.vuanem.com/186x186,webp,q80...",https://vuanem.com/nem-cao-su-gummi-classic.ht...,Việt Nam,180.0,Cứng vừa,5.0,6712000,120.0,200.0
1,Nệm cao su thiên nhiên Gummi Classic thế hệ mớ...,Gummi,Cao su thiên nhiên,Cao su,"Thương hiệu Gummi - Tinh khiết từ cao su, đúng...",4.9,359.0,44410.0,"https://cdn-cf-ops.vuanem.com/186x186,webp,q80...",https://vuanem.com/nem-cao-su-gummi-classic.ht...,Việt Nam,180.0,Cứng vừa,10.0,7808000,120.0,200.0
2,Nệm cao su thiên nhiên Gummi Classic thế hệ mớ...,Gummi,Cao su thiên nhiên,Cao su,"Thương hiệu Gummi - Tinh khiết từ cao su, đúng...",4.9,359.0,44410.0,"https://cdn-cf-ops.vuanem.com/186x186,webp,q80...",https://vuanem.com/nem-cao-su-gummi-classic.ht...,Việt Nam,180.0,Cứng vừa,15.0,12608000,120.0,200.0
3,Nệm cao su thiên nhiên Gummi Classic thế hệ mớ...,Gummi,Cao su thiên nhiên,Cao su,"Thương hiệu Gummi - Tinh khiết từ cao su, đúng...",4.9,359.0,44410.0,"https://cdn-cf-ops.vuanem.com/186x186,webp,q80...",https://vuanem.com/nem-cao-su-gummi-classic.ht...,Việt Nam,180.0,Cứng vừa,5.0,7352000,140.0,200.0
4,Nệm cao su thiên nhiên Gummi Classic thế hệ mớ...,Gummi,Cao su thiên nhiên,Cao su,"Thương hiệu Gummi - Tinh khiết từ cao su, đúng...",4.9,359.0,44410.0,"https://cdn-cf-ops.vuanem.com/186x186,webp,q80...",https://vuanem.com/nem-cao-su-gummi-classic.ht...,Việt Nam,180.0,Cứng vừa,10.0,9408000,140.0,200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4527,Nệm Cao Su Tổng Hợp Kim Cương,Kim Cương,Cao su tổng hợp,Foam,NỆM CAO SU TỔNG HỢP KIM CƯƠNG Nệm cao su tổng ...,NaN,NaN,NaN,https://bizweb.dktcdn.net/thumb/medium/100/405...,https://thegioigiuongnem.com/nem-cao-su-tong-h...,Việt Nam,120.0,Mềm - Trung bình,22.0,3175000,100.0,200.0
4528,Nệm Cao Su Tổng Hợp Kim Cương,Kim Cương,Cao su tổng hợp,Foam,NỆM CAO SU TỔNG HỢP KIM CƯƠNG Nệm cao su tổng ...,NaN,NaN,NaN,https://bizweb.dktcdn.net/thumb/medium/100/405...,https://thegioigiuongnem.com/nem-cao-su-tong-h...,Việt Nam,120.0,Mềm - Trung bình,22.0,3377000,120.0,200.0
4529,Nệm Cao Su Tổng Hợp Kim Cương,Kim Cương,Cao su tổng hợp,Foam,NỆM CAO SU TỔNG HỢP KIM CƯƠNG Nệm cao su tổng ...,NaN,NaN,NaN,https://bizweb.dktcdn.net/thumb/medium/100/405...,https://thegioigiuongnem.com/nem-cao-su-tong-h...,Việt Nam,120.0,Mềm - Trung bình,22.0,3578000,140.0,200.0
4530,Nệm Cao Su Tổng Hợp Kim Cương,Kim Cương,Cao su tổng hợp,Foam,NỆM CAO SU TỔNG HỢP KIM CƯƠNG Nệm cao su tổng ...,NaN,NaN,NaN,https://bizweb.dktcdn.net/thumb/medium/100/405...,https://thegioigiuongnem.com/nem-cao-su-tong-h...,Việt Nam,120.0,Mềm - Trung bình,22.0,3780000,160.0,200.0


In [4]:
df['firmness'] = df['firmness'].fillna('Không xác định')

df['rating']               = df['rating'].fillna(0)
df['reviews']              = df['reviews'].fillna(0)
df['product_sold_number']  = df['product_sold_number'].fillna(0)

df = df.dropna(subset=['description', 'brand', 
                        'width', 'length', 
                        'thickness', 'price'])

print(f"Shape sau xử lý null: {df.shape}")
print("\nNull rate còn lại:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape sau xử lý null: (4457, 17)

Null rate còn lại:
origin       6
warranty    93
dtype: int64


In [5]:
df = df.dropna(subset=['origin'])

df['warranty'] = df['warranty'].fillna(
    df['warranty'].median()
)

# Kiểm tra
print(f"Shape: {df.shape}")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape: (4451, 17)
Series([], dtype: int64)


In [6]:
df['price'] = df['price'].astype(float)

### Lựa chọn max_features cho TF-IDF

Dựa trên phân tích EDA:
- Median độ dài description = 490 từ
- Mean = 652 từ

Chọn `max_features=1000` vì:
- Gấp đôi median → đủ capture hầu hết 
  vocabulary quan trọng
- Bao phủ các từ kỹ thuật đặc thù 
  ngành nệm (chất liệu, công nghệ, 
  tính năng...)

In [7]:
tfidf_desc = TfidfVectorizer(
    max_features=1000,   
    ngram_range=(1, 2),
    min_df=2,
    strip_accents=None,
    analyzer='word'
)

In [8]:
desc_matrix = tfidf_desc.fit_transform(
    df['description']
)
print(f"Description matrix: {desc_matrix.shape}")

Description matrix: (4451, 1000)


In [9]:
tfidf_mat = TfidfVectorizer(
    max_features=50,
    ngram_range=(1, 2),
    min_df=1,
    strip_accents=None
)
mat_matrix = tfidf_mat.fit_transform(
    df['material_type']
)
print(f"Material matrix: {mat_matrix.shape}")

Material matrix: (4451, 48)


In [10]:
df['firmness'].value_counts()

firmness
Trung bình               1548
Cứng                      958
Cứng vừa                  721
Mềm - Trung bình          623
Trung bình - Hơi cứng     324
Mềm                       141
Không xác định            136
Name: count, dtype: int64

In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# Định nghĩa thứ tự
firmness_order = [
    'Mềm',
    'Mềm - Trung bình', 
    'Trung bình',
    'Trung bình - Hơi cứng',
    'Cứng vừa',
    'Cứng',
    'Không xác định'
]

oe = OrdinalEncoder(
    categories=[firmness_order],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

firmness_encoded = oe.fit_transform(df[['firmness']])
print(f"Firmness encoded shape: {firmness_encoded.shape}")

ohe = OneHotEncoder(sparse_output=True,
                    handle_unknown='ignore')
brand_matrix = ohe.fit_transform(df[['brand']])
print(f"Brand matrix: {brand_matrix.shape}")

Firmness encoded shape: (4451, 1)
Brand matrix: (4451, 31)


In [12]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Log transform trước
df['price_log']   = np.log1p(df['price'])
df['reviews_log'] = np.log1p(df['reviews'])
df['sold_log']    = np.log1p(df['product_sold_number'])

# Scale
scaler = StandardScaler()
num_cols = [
    'price_log', 
    'reviews_log', 
    'sold_log',
    'rating',
    'warranty', 
    'thickness', 
    'width', 
    'length'
]

num_matrix = scaler.fit_transform(df[num_cols])
print(f"Numerical matrix: {num_matrix.shape}")

Numerical matrix: (4451, 8)


In [13]:
feature_matrix = sp.hstack([
    desc_matrix,                      # (4451, 1000) - TF-IDF description
    mat_matrix,                       # (4451, 48)   - TF-IDF material_type
    brand_matrix,                     # (4451, ~35)  - One-hot brand
    sp.csr_matrix(firmness_encoded),  # (4451, 1)    - Ordinal firmness
    sp.csr_matrix(num_matrix)         # (4451, 8)    - Numerical features
]).tocsr()

print(f"Feature matrix shape: {feature_matrix.shape}")

Feature matrix shape: (4451, 1088)


In [14]:
def calculate_popularity(row):
    score = 0
    count = 0
    
    if row['rating'] > 0:
        score += (row['rating'] / 5) * 40
        count += 1
    if row['reviews'] > 0:
        score += np.log1p(row['reviews']) * 30
        count += 1
    if row['product_sold_number'] > 0:
        score += np.log1p(row['product_sold_number']) * 30
        count += 1
    
    return score if count > 0 else 0

df['popularity_score'] = df.apply(calculate_popularity, axis=1)

print(f"Popularity score:")
print(df['popularity_score'].describe())

Popularity score:
count    4451.000000
mean      183.446592
std       189.513049
min         0.000000
25%         0.000000
50%       111.936858
75%       386.195598
max       555.698405
Name: popularity_score, dtype: float64


In [15]:
# Lưu feature matrix
sp.save_npz('../data/final/feature_matrix.npz', feature_matrix)

# Lưu dataframe
df.to_csv('../data/final/mattresses_features.csv',
          index=False, encoding='utf-8-sig')

# Lưu các encoder/scaler để dùng trong app
pickle.dump(tfidf_desc, open('../models/tfidf_desc.pkl', 'wb'))
pickle.dump(tfidf_mat,  open('../models/tfidf_mat.pkl',  'wb'))
pickle.dump(ohe,        open('../models/ohe_brand.pkl',  'wb'))
pickle.dump(oe,         open('../models/oe_firmness.pkl','wb'))
pickle.dump(scaler,     open('../models/scaler.pkl',     'wb'))

# Lưu index để map kết quả về sản phẩm
df[['product_name', 'brand', 'category', 
    'material_type', 'firmness', 'price',
    'width', 'length', 'thickness',
    'image_url', 'link', 
    'popularity_score']].to_csv(
        '../data/final/mattresses_index.csv',
        index=True,   # giữ index để map với feature matrix
        encoding='utf-8-sig'
)

print(f"\n Đã lưu tất cả")
print(f"Feature matrix:      data/final/feature_matrix.npz  {feature_matrix.shape}")
print(f"Features CSV:        data/final/mattresses_features.csv")
print(f"Index CSV:           data/final/mattresses_index.csv")
print(f"TF-IDF description:  models/tfidf_desc.pkl")
print(f"TF-IDF material:     models/tfidf_mat.pkl")
print(f"OHE brand:           models/ohe_brand.pkl")
print(f"Ordinal firmness:    models/oe_firmness.pkl")
print(f"Scaler:              models/scaler.pkl")


 Đã lưu tất cả
Feature matrix:      data/final/feature_matrix.npz  (4451, 1088)
Features CSV:        data/final/mattresses_features.csv
Index CSV:           data/final/mattresses_index.csv
TF-IDF description:  models/tfidf_desc.pkl
TF-IDF material:     models/tfidf_mat.pkl
OHE brand:           models/ohe_brand.pkl
Ordinal firmness:    models/oe_firmness.pkl
Scaler:              models/scaler.pkl
